<a href="https://colab.research.google.com/github/icmsol/capstone-project-7-integrated-ai-systems-synthesis/blob/main/notebooks/Project7_Integration_Smoke_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 7 — Integration and Asset Smoke Tests

**Configurable Small-Business Opportunity-to-Contract Intelligence and Assurance Framework**  
**ICM Solutions Reference Implementation**

## P1-05 objectives

This notebook performs read-only and controlled smoke tests before system architecture and implementation begin. It validates:

1. the Project 7 repository structure;
2. the ICM and fictional organization profiles;
3. profile-linked JSON and CSV configuration artifacts;
4. fixed responsible-AI safeguards and prohibited profile overrides;
5. the Project 4 CPU inference package;
6. selected frozen Project 2 historical-analysis assets;
7. the Project 6 official FAR source-acquisition baseline;
8. a durable pass, warning, and failure diagnostic record.

## Boundary

This notebook does **not** make a bid/no-bid decision, provide legal advice, approve compliance, accept contract terms, commit staffing, communicate externally, or execute a recommendation. It verifies component readiness only.


## 1. Install and import the minimal smoke-test dependencies

The notebook uses the standard Colab CPU runtime. No T4 is required because the Project 4 checkpoint has already been trained and exported.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "jsonschema": "jsonschema",
    "bs4": "beautifulsoup4",
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )

print("Minimal dependencies are available.")


In [ ]:
from __future__ import annotations

import csv
import gzip
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from bs4 import BeautifulSoup
from jsonschema import Draft202012Validator

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Selected smoke-test device: cpu")


## 2. Clone or refresh the authoritative Project 7 repository

The notebook uses the public GitHub repository as the source of truth. It does not require a GitHub credential for read-only cloning.

In [ ]:
PROJECT7_GIT_URL = (
    "https://github.com/icmsol/"
    "capstone-project-7-integrated-ai-systems-synthesis.git"
)
PROJECT7_REPO_DIR = Path(
    "/content/capstone-project-7-integrated-ai-systems-synthesis"
)
WORK_DIR = Path("/content/project7_p1_05_work")
AUDIT_OUTPUT_DIR = WORK_DIR / "audit"
PRIOR_ASSET_DIR = WORK_DIR / "prior_assets"

if PROJECT7_REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT7_REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", PROJECT7_GIT_URL, str(PROJECT7_REPO_DIR)],
        check=True,
    )

AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_ASSET_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT7_REPO_DIR)

commit_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

print(f"Repository: {PROJECT7_REPO_DIR}")
print(f"Commit SHA: {commit_sha}")


## 3. Diagnostic helpers

Every test records a stable ID, status, summary, and evidence. A warning identifies an unresolved dependency without falsely presenting it as a pass.

In [ ]:
diagnostics: list[dict[str, Any]] = []


def record_result(
    test_id: str,
    status: str,
    summary: str,
    evidence: dict[str, Any] | None = None,
) -> None:
    allowed_statuses = {"PASS", "WARN", "FAIL"}

    if status not in allowed_statuses:
        raise ValueError(
            f"Unsupported status {status!r}; expected one of {sorted(allowed_statuses)}."
        )

    diagnostics.append(
        {
            "test_id": test_id,
            "status": status,
            "summary": summary,
            "evidence": evidence or {},
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        }
    )

    print(f"{status}: {test_id} — {summary}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as source_file:
        for block in iter(lambda: source_file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as source_file:
        return json.load(source_file)


def resolve_profile_reference(
    profile_path: Path,
    reference: str,
) -> Path:
    return (profile_path.parent / reference).resolve()


print("Diagnostic helpers are ready.")


## 4. Validate the repository structure and required artifacts

In [ ]:
required_directories = [
    "audit",
    "config",
    "data",
    "docs",
    "figures",
    "models/project4",
    "notebooks",
    "outputs",
    "presentation",
    "reports",
    "src",
    "tests",
]

required_files = [
    "README.md",
    "config/profiles/icm_solutions.json",
    "config/profiles/icm_service_catalog.csv",
    "config/profiles/fictional_small_business.json",
    "config/profiles/fictional_service_catalog.csv",
    "config/schemas/organization_profile.schema.json",
    "config/system/fixed_safeguards.json",
    "docs/Project_1_2_Reusable_Asset_Review.md",
    "docs/Project_4_Reusable_Asset_Review.md",
    "docs/Project_6_Reusable_Asset_Review.md",
    "models/project4/selected_clause_classifier.pt",
    "models/project4/index_to_token.json",
    "models/project4/token_to_index.json",
    "models/project4/label_id_to_category.json",
    "models/project4/model_config.json",
    "models/project4/tokenizer_config.json",
    "models/project4/validation_selection_metrics.json",
    "audit/project4_cpu_smoke_test.json",
    "audit/project4_inference_manifest.json",
]

missing_directories = [
    relative_path
    for relative_path in required_directories
    if not (PROJECT7_REPO_DIR / relative_path).is_dir()
]

missing_files = [
    relative_path
    for relative_path in required_files
    if not (PROJECT7_REPO_DIR / relative_path).is_file()
]

if missing_directories or missing_files:
    record_result(
        "P1-05-T01",
        "FAIL",
        "Repository structure or required artifacts are incomplete.",
        {
            "missing_directories": missing_directories,
            "missing_files": missing_files,
        },
    )
    raise RuntimeError(
        "Repository validation failed. Review the recorded missing paths."
    )

record_result(
    "P1-05-T01",
    "PASS",
    "Repository structure and required starter artifacts are present.",
    {
        "required_directory_count": len(required_directories),
        "required_file_count": len(required_files),
        "commit_sha": commit_sha,
    },
)


## 5. Validate organization profiles, linked configuration, and service catalogs

Both profiles must use the same schema and loading logic. Organization-specific differences are permitted; broken references and invalid structures are not.

In [ ]:
schema_path = (
    PROJECT7_REPO_DIR
    / "config/schemas/organization_profile.schema.json"
)
organization_schema = load_json(schema_path)
profile_validator = Draft202012Validator(organization_schema)

profile_paths = {
    "ICMSOL": (
        PROJECT7_REPO_DIR
        / "config/profiles/icm_solutions.json"
    ),
    "RCALABS": (
        PROJECT7_REPO_DIR
        / "config/profiles/fictional_small_business.json"
    ),
}

required_profile_references = [
    "service_catalog_file",
    "opportunity_rules_file",
    "staffing_map_file",
    "reviewer_roles_file",
    "recommendation_thresholds_file",
    "fixed_safeguards_file",
]

required_service_columns = {
    "organization_id",
    "organization_name",
    "profile_version",
    "service_family_id",
    "service_family",
    "capability_id",
    "capability_name",
    "capability_description",
    "positive_keywords",
    "strong_match_phrases",
    "exclusion_keywords",
    "default_priority",
    "active",
}

loaded_profiles: dict[str, dict[str, Any]] = {}
service_catalogs: dict[str, pd.DataFrame] = {}
profile_validation_evidence: dict[str, Any] = {}

for expected_id, profile_path in profile_paths.items():
    profile = load_json(profile_path)
    errors = sorted(
        profile_validator.iter_errors(profile),
        key=lambda error: list(error.path),
    )

    if errors:
        error_messages = [
            {
                "path": list(error.path),
                "message": error.message,
            }
            for error in errors
        ]
        record_result(
            "P1-05-T02",
            "FAIL",
            f"Organization profile {expected_id} failed schema validation.",
            {"errors": error_messages},
        )
        raise RuntimeError(
            f"Schema validation failed for {profile_path.name}."
        )

    if profile["organization_id"] != expected_id:
        raise RuntimeError(
            f"Profile ID mismatch: expected {expected_id}, "
            f"found {profile['organization_id']}."
        )

    resolved_references = {}

    for reference_key in required_profile_references:
        referenced_path = resolve_profile_reference(
            profile_path,
            profile[reference_key],
        )

        if not referenced_path.is_file():
            raise FileNotFoundError(
                f"{expected_id} profile reference is missing: "
                f"{reference_key} -> {referenced_path}"
            )

        resolved_references[reference_key] = str(
            referenced_path.relative_to(PROJECT7_REPO_DIR)
        )

    service_catalog_path = resolve_profile_reference(
        profile_path,
        profile["service_catalog_file"],
    )
    service_catalog = pd.read_csv(service_catalog_path)

    missing_service_columns = required_service_columns.difference(
        service_catalog.columns
    )

    if missing_service_columns:
        raise ValueError(
            f"{expected_id} service catalog is missing columns: "
            f"{sorted(missing_service_columns)}"
        )

    active_services = service_catalog.loc[
        service_catalog["active"].astype(str).str.upper() == "TRUE"
    ].copy()

    if active_services.empty:
        raise ValueError(
            f"{expected_id} has no active service capabilities."
        )

    if active_services["capability_id"].duplicated().any():
        raise ValueError(
            f"{expected_id} contains duplicate capability IDs."
        )

    loaded_profiles[expected_id] = profile
    service_catalogs[expected_id] = active_services
    profile_validation_evidence[expected_id] = {
        "profile_version": profile["profile_version"],
        "fictional": profile["fictional"],
        "active_capabilities": int(len(active_services)),
        "service_families": int(
            active_services["service_family_id"].nunique()
        ),
        "resolved_references": resolved_references,
    }

record_result(
    "P1-05-T02",
    "PASS",
    "Both organization profiles and their linked configuration artifacts are valid.",
    profile_validation_evidence,
)

display(
    pd.DataFrame(profile_validation_evidence)
    .T
    .reset_index(names="organization_id")
)


## 6. Validate fixed safeguard invariance

An organization profile may tailor business criteria but may not disable evidence traceability, abstention, audit logging, human approval, privacy controls, or the prohibition on autonomous external actions.

In [ ]:
fixed_safeguard_path = (
    PROJECT7_REPO_DIR
    / "config/system/fixed_safeguards.json"
)
fixed_safeguards = load_json(fixed_safeguard_path)

if fixed_safeguards.get("ordinary_profile_override_permitted") is not False:
    raise RuntimeError(
        "Fixed safeguard policy unexpectedly permits ordinary profile overrides."
    )

prohibited_profile_keys = set(
    fixed_safeguards.get("prohibited_profile_keys", [])
)

if not prohibited_profile_keys:
    raise RuntimeError(
        "No prohibited profile-override keys were defined."
    )

profile_override_attempts: dict[str, list[str]] = {}

for organization_id, profile in loaded_profiles.items():
    serialized_profile = json.dumps(profile)
    attempted_keys = sorted(
        key
        for key in prohibited_profile_keys
        if key in serialized_profile
    )
    profile_override_attempts[organization_id] = attempted_keys

if any(profile_override_attempts.values()):
    record_result(
        "P1-05-T03",
        "FAIL",
        "An organization profile attempts to override a fixed safeguard.",
        profile_override_attempts,
    )
    raise RuntimeError(
        "Fixed-safeguard invariance validation failed."
    )

required_fixed_controls = [
    "evidence_traceability_required",
    "counterevidence_required",
    "abstention_required_when_evidence_insufficient",
    "human_final_decision_required",
    "audit_logging_required",
    "secrets_prohibited_in_repository",
    "autonomous_external_actions_prohibited",
    "unsupported_recommendations_prohibited",
]

control_values = fixed_safeguards.get("controls", {})
disabled_required_controls = [
    control
    for control in required_fixed_controls
    if control_values.get(control) is not True
]

if disabled_required_controls:
    raise RuntimeError(
        "Required fixed safeguards are missing or disabled: "
        f"{disabled_required_controls}"
    )

record_result(
    "P1-05-T03",
    "PASS",
    "Organization profiles cannot disable the required framework safeguards.",
    {
        "prohibited_override_key_count": len(prohibited_profile_keys),
        "validated_required_controls": required_fixed_controls,
    },
)


## 7. Verify the same loader produces organization-specific capability context

This is an initial portability smoke test, not the final recommendation evaluation.

In [ ]:
def catalog_terms(
    catalog: pd.DataFrame,
) -> set[str]:
    terms: set[str] = set()

    for column in [
        "capability_name",
        "positive_keywords",
        "strong_match_phrases",
    ]:
        for value in catalog[column].fillna("").astype(str):
            for term in re.split(r"[;|]", value):
                normalized = re.sub(
                    r"\s+",
                    " ",
                    term.strip().lower(),
                )
                if normalized:
                    terms.add(normalized)

    return terms


icm_terms = catalog_terms(service_catalogs["ICMSOL"])
fictional_terms = catalog_terms(service_catalogs["RCALABS"])

unique_icm_terms = sorted(icm_terms - fictional_terms)
unique_fictional_terms = sorted(fictional_terms - icm_terms)

if not unique_icm_terms or not unique_fictional_terms:
    raise RuntimeError(
        "The portability profiles do not expose distinct capability context."
    )

record_result(
    "P1-05-T04",
    "PASS",
    "The shared profile loader exposes distinct organization-specific capability context.",
    {
        "icm_unique_term_count": len(unique_icm_terms),
        "fictional_unique_term_count": len(unique_fictional_terms),
        "icm_examples": unique_icm_terms[:10],
        "fictional_examples": unique_fictional_terms[:10],
        "source_code_change_required": False,
    },
)


## 8. Load and smoke-test the Project 4 classifier on CPU

The model supports bounded clause-theme triage only. Its output is not legal interpretation or contract acceptance.

In [ ]:
@dataclass(frozen=True)
class TransformerModelConfig:
    vocab_size: int
    num_classes: int
    max_length: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    feedforward_dim: int
    dropout: float
    pad_index: int


class TransformerClauseClassifier(nn.Module):
    def __init__(
        self,
        config: TransformerModelConfig,
    ) -> None:
        super().__init__()

        if config.embedding_dim % config.num_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_heads."
            )

        self.config = config
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.embedding_dim,
            padding_idx=config.pad_index,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=config.max_length,
            embedding_dim=config.embedding_dim,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.embedding_dim,
            nhead=config.num_heads,
            dim_feedforward=config.feedforward_dim,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=config.num_layers,
            norm=nn.LayerNorm(config.embedding_dim),
        )
        self.output_dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(
            config.embedding_dim,
            config.num_classes,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, sequence_length = input_ids.shape

        if sequence_length > self.config.max_length:
            raise ValueError(
                f"Input length {sequence_length} exceeds configured "
                f"maximum {self.config.max_length}."
            )

        position_ids = (
            torch.arange(
                sequence_length,
                device=input_ids.device,
            )
            .unsqueeze(0)
            .expand(batch_size, -1)
        )

        hidden_states = (
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

        key_padding_mask = ~attention_mask.bool()
        encoded_states = self.encoder(
            hidden_states,
            src_key_padding_mask=key_padding_mask,
        )

        valid_token_mask = (
            attention_mask.unsqueeze(-1)
            .to(encoded_states.dtype)
        )

        pooled_states = (
            (encoded_states * valid_token_mask).sum(dim=1)
            / valid_token_mask.sum(dim=1).clamp(min=1.0)
        )

        return self.classifier(
            self.output_dropout(pooled_states)
        )


project4_dir = PROJECT7_REPO_DIR / "models/project4"

model_config = load_json(
    project4_dir / "model_config.json"
)
tokenizer_config = load_json(
    project4_dir / "tokenizer_config.json"
)
token_to_index = load_json(
    project4_dir / "token_to_index.json"
)
label_id_to_category_raw = load_json(
    project4_dir / "label_id_to_category.json"
)
label_id_to_category = {
    int(label_id): category
    for label_id, category in label_id_to_category_raw.items()
}

model = TransformerClauseClassifier(
    TransformerModelConfig(**model_config)
)
state_dictionary = torch.load(
    project4_dir / "selected_clause_classifier.pt",
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(state_dictionary)
model.eval()

token_pattern = re.compile(
    tokenizer_config["pattern"]
)
maximum_length = int(
    tokenizer_config["maximum_sequence_length"]
)
pad_index = int(
    tokenizer_config["pad_index"]
)
unk_index = int(
    tokenizer_config["unk_index"]
)

sample_clause = (
    "The contractor shall maintain records and permit the Government "
    "to inspect and audit those records upon reasonable notice."
)

tokens = token_pattern.findall(
    sample_clause.lower()
)
token_ids = [
    int(token_to_index.get(token, unk_index))
    for token in tokens[:maximum_length]
]
attention = [True] * len(token_ids)

padding_count = maximum_length - len(token_ids)
token_ids.extend([pad_index] * padding_count)
attention.extend([False] * padding_count)

input_ids = torch.tensor(
    [token_ids],
    dtype=torch.long,
)
attention_mask = torch.tensor(
    [attention],
    dtype=torch.bool,
)

with torch.inference_mode():
    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )
    probabilities = torch.softmax(
        logits,
        dim=-1,
    )

if tuple(logits.shape) != (
    1,
    model_config["num_classes"],
):
    raise RuntimeError(
        f"Unexpected classifier output shape: {tuple(logits.shape)}"
    )

if not torch.isfinite(probabilities).all():
    raise RuntimeError(
        "Project 4 classifier produced nonfinite probabilities."
    )

probability_sum = float(
    probabilities.sum().item()
)

if abs(probability_sum - 1.0) > 1e-6:
    raise RuntimeError(
        f"Classifier probabilities do not sum to one: {probability_sum}"
    )

predicted_label_id = int(
    probabilities.argmax(dim=-1).item()
)
predicted_category = label_id_to_category[
    predicted_label_id
]
prediction_confidence = float(
    probabilities[0, predicted_label_id].item()
)

record_result(
    "P1-05-T05",
    "PASS",
    "The Project 4 inference package loads and produces a finite CPU prediction.",
    {
        "checkpoint_sha256": sha256_file(
            project4_dir / "selected_clause_classifier.pt"
        ),
        "prediction": predicted_category,
        "confidence": prediction_confidence,
        "probability_sum": probability_sum,
        "output_shape": list(logits.shape),
        "device": "cpu",
        "boundary": (
            "Clause-theme triage only; mandatory human review "
            "for consequential interpretation."
        ),
    },
)

print(f"Predicted category: {predicted_category}")
print(f"Confidence: {prediction_confidence:.6f}")


## 9. Load selected Project 2 frozen historical assets

These assets provide descriptive historical context. They are not forecasts or award-probability evidence.

In [ ]:
PROJECT2_RAW_BASE = (
    "https://raw.githubusercontent.com/icmsol/"
    "Capstone-project-2-statistical-analysis/main/data/processed"
)

project2_assets = {
    "analysis_summary.json": (
        f"{PROJECT2_RAW_BASE}/analysis_summary.json"
    ),
    "data_dictionary.csv": (
        f"{PROJECT2_RAW_BASE}/data_dictionary.csv"
    ),
    "classification_audit_sample.csv": (
        f"{PROJECT2_RAW_BASE}/classification_audit_sample.csv"
    ),
}

project2_dir = PRIOR_ASSET_DIR / "project2"
project2_dir.mkdir(parents=True, exist_ok=True)

download_evidence = {}

for filename, url in project2_assets.items():
    response = requests.get(
        url,
        timeout=120,
    )
    response.raise_for_status()

    target_path = project2_dir / filename
    target_path.write_bytes(response.content)

    download_evidence[filename] = {
        "url": url,
        "bytes": target_path.stat().st_size,
        "sha256": sha256_file(target_path),
    }

analysis_summary = load_json(
    project2_dir / "analysis_summary.json"
)
data_dictionary = pd.read_csv(
    project2_dir / "data_dictionary.csv"
)
classification_audit = pd.read_csv(
    project2_dir / "classification_audit_sample.csv"
)

if data_dictionary.empty or classification_audit.empty:
    raise RuntimeError(
        "One or more Project 2 tabular assets are empty."
    )

record_result(
    "P1-05-T06",
    "PASS",
    "Selected Project 2 frozen historical assets loaded successfully.",
    {
        "downloaded_assets": download_evidence,
        "data_dictionary_rows": int(len(data_dictionary)),
        "classification_audit_rows": int(len(classification_audit)),
        "analysis_summary_top_level_keys": sorted(
            analysis_summary.keys()
        ),
        "limitation": (
            "Descriptive historical context only; not an award, "
            "eligibility, staffing-capacity, or bid/no-bid predictor."
        ),
    },
)

display(data_dictionary.head())


## 10. Reacquire and validate the Project 6 official FAR source baseline

The original Project 6 notebook acquired the official Acquisition.gov FAR Subpart 52.2 page, confirmed FAC 2026-01 and the March 13, 2026 effective date, and froze the source. This smoke test checks whether the same official source baseline remains available.

A byte-for-byte match is recorded as an exact recovery. The same FAC metadata with different bytes is a warning and must be treated as a newly acquired candidate—not silently labeled as the original frozen file.

In [ ]:
FAR_SOURCE_URL = (
    "https://www.acquisition.gov/far/subpart-52.2"
)
EXPECTED_FAC_NUMBER = "2026-01"
EXPECTED_EFFECTIVE_DATE = "03/13/2026"
EXPECTED_PROJECT6_RAW_SHA256 = (
    "58cdd8faddf9a6b591b81afaf03639c350da795ca57f233df5b96a05962a54fd"
)

far_response = requests.get(
    FAR_SOURCE_URL,
    headers={
        "User-Agent": (
            "ICM-Capstone-Project-7/1.0 "
            "(academic reproducibility and public-source research)"
        ),
        "Accept": "text/html,application/xhtml+xml",
    },
    timeout=120,
)
far_response.raise_for_status()
far_html = far_response.content

if len(far_html) < 500_000:
    raise RuntimeError(
        "The official FAR source response is unexpectedly small: "
        f"{len(far_html):,} bytes."
    )

far_soup = BeautifulSoup(
    far_html,
    "html.parser",
)
far_visible_text = re.sub(
    r"\s+",
    " ",
    far_soup.get_text(
        " ",
        strip=True,
    ),
)

required_source_markers = [
    EXPECTED_FAC_NUMBER,
    EXPECTED_EFFECTIVE_DATE,
    "Subpart 52.2",
    "52.200 Scope of subpart",
]

missing_source_markers = [
    marker
    for marker in required_source_markers
    if marker not in far_visible_text
]

if missing_source_markers:
    record_result(
        "P1-05-T07",
        "FAIL",
        "The official FAR source does not match the expected metadata baseline.",
        {
            "missing_markers": missing_source_markers,
            "source_url": FAR_SOURCE_URL,
        },
    )
    raise RuntimeError(
        "Official FAR source validation failed."
    )

far_raw_sha256 = hashlib.sha256(
    far_html
).hexdigest()

far_recovery_dir = WORK_DIR / "far_recovery"
far_recovery_dir.mkdir(
    parents=True,
    exist_ok=True,
)
far_gzip_path = (
    far_recovery_dir
    / "far_subpart_52_2_fac_2026_01.html.gz"
)

with gzip.open(
    far_gzip_path,
    "wb",
    compresslevel=9,
) as compressed_file:
    compressed_file.write(far_html)

exact_project6_match = (
    far_raw_sha256
    == EXPECTED_PROJECT6_RAW_SHA256
)

far_status = (
    "PASS"
    if exact_project6_match
    else "WARN"
)

far_summary = (
    "The official FAR source is a byte-for-byte match to the "
    "Project 6 raw-source checksum."
    if exact_project6_match
    else (
        "The official FAR source has the expected FAC metadata but "
        "does not match the Project 6 byte checksum. Treat it as a "
        "newly acquired candidate until the difference is resolved."
    )
)

record_result(
    "P1-05-T07",
    far_status,
    far_summary,
    {
        "source_url": FAR_SOURCE_URL,
        "fac_number": EXPECTED_FAC_NUMBER,
        "effective_date": EXPECTED_EFFECTIVE_DATE,
        "retrieved_bytes": len(far_html),
        "retrieved_sha256": far_raw_sha256,
        "expected_project6_sha256": EXPECTED_PROJECT6_RAW_SHA256,
        "exact_project6_match": exact_project6_match,
        "local_gzip_path": str(far_gzip_path),
        "http_status": far_response.status_code,
        "content_type": far_response.headers.get("Content-Type"),
    },
)


## 11. Write the P1-05 diagnostic record

The JSON output is generated outside the cloned repository. After review, an approved copy may be committed under `audit/`.

In [ ]:
status_counts = {
    status: sum(
        result["status"] == status
        for result in diagnostics
    )
    for status in ["PASS", "WARN", "FAIL"]
}

diagnostic_record = {
    "artifact_id": "PROJECT7-P1-05-INTEGRATION-SMOKE-TEST",
    "artifact_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project7_repository": PROJECT7_GIT_URL,
    "project7_commit_sha": commit_sha,
    "runtime": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "device": "cpu",
    },
    "status_counts": status_counts,
    "overall_status": (
        "FAIL"
        if status_counts["FAIL"]
        else (
            "PASS_WITH_WARNING"
            if status_counts["WARN"]
            else "PASS"
        )
    ),
    "tests": diagnostics,
    "production_boundary": (
        "Controlled capstone smoke test only; no production-readiness "
        "claim and no autonomous external action."
    ),
}

diagnostic_path = (
    AUDIT_OUTPUT_DIR
    / "p1_05_integration_smoke_test.json"
)

with diagnostic_path.open(
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        diagnostic_record,
        output_file,
        indent=2,
    )

summary_table = pd.DataFrame(
    [
        {
            "test_id": item["test_id"],
            "status": item["status"],
            "summary": item["summary"],
        }
        for item in diagnostics
    ]
)

display(summary_table)

print(
    "Overall status:",
    diagnostic_record["overall_status"],
)
print(
    "Diagnostic file:",
    diagnostic_path,
)
print(
    "PASS / WARN / FAIL:",
    status_counts,
)

if status_counts["FAIL"]:
    raise RuntimeError(
        "One or more P1-05 smoke tests failed. "
        "Do not proceed to architecture until resolved."
    )


## Expected completion state

A successful run should show:

- no missing Project 7 repository artifacts;
- valid ICM and fictional profiles;
- fixed safeguard invariance;
- distinct organization capability contexts;
- a finite Project 4 CPU prediction;
- successful loading of selected Project 2 assets;
- either:
  - an exact Project 6 FAR source checksum match; or
  - a clearly labeled warning requiring source-difference review.

Do not commit the generated audit file until the outputs have been reviewed.